In [ ]:
# ======================================================
# RF + BiLSTM residual corrector
# Optimised by PROPER Bitterling Colony Optimization (BCO)
#
# BCO Reference:
#   Lai Z, Ding J, Yin H, Wu L, Liu J, He W.
#   "Bitterling Colony Optimization: a bio-inspired algorithm
#    for global search." Cluster Computing (2025) 28:542
#   https://doi.org/10.1007/s10586-025-05170-x
#
# Three bitterling behaviours implemented EXACTLY as in the paper:
#   RB  — Reproductive Behaviour  (Eq. 10, 12)
#   EB  — Eviction Behaviour      (Eq. 14–17) + DPCM counting matrix
#   TB  — Travel Behaviour        (Eq. 11, 12)
#
# Parameters (paper Table 4 / Sect. 4.1.1 recommendations):
#   Pdis = 0.7   (probability male finds & reaches a mussel)
#   Pl   = 0.9   (probability of being evicted)
#   Net  = 200   (number of dimension splits for DPCM)
#   N    must be even (first N/2 = males, second N/2 = females)
# ======================================================

# ======================================================
# 1. Imports
# ======================================================
import pandas as pd
import numpy as np
import os, gc, warnings, json, signal, traceback

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras import backend as K

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from scipy.stats import wilcoxon
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ======================================================
# 2. Global seeds
# ======================================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# ======================================================
# 3. GPU
# ======================================================
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        tf.config.set_visible_devices(gpus[0], 'GPU')
        _name = tf.config.experimental.get_device_details(gpus[0]).get('device_name', 'GPU')
        print(f"✅ GPU: {_name}")
        strategy = tf.distribute.OneDeviceStrategy("/gpu:0")
    except RuntimeError as e:
        print("⚠️ GPU setup failed:", e)
        strategy = tf.distribute.get_strategy()
else:
    print("ℹ️ CPU only")
    strategy = tf.distribute.get_strategy()

# ======================================================
# 4. Load raw data
# ======================================================
FILE_PATH = ("/mnt/f/Solar-Review-Ppaer/Monthwise-Data-Sheets/"
             "Data_clean/10-Minute-Average-Data/10-min-AVG-Data.xlsx")

raw = pd.read_excel(FILE_PATH)
raw = raw[['DateTime', 'Power_MW']]
raw['Power_MW'] = raw['Power_MW'].fillna(0)
raw['DateTime'] = pd.to_datetime(raw['DateTime'])
raw.set_index('DateTime', inplace=True)

# ======================================================
# 5. Strict temporal split — 70% train | 15% val | 15% test
# ======================================================
N_data = len(raw)
n_tr   = int(N_data * 0.70)
n_val  = int(N_data * 0.15)

raw_train = raw.iloc[:n_tr].copy()
raw_val   = raw.iloc[n_tr:n_tr + n_val].copy()
raw_test  = raw.iloc[n_tr + n_val:].copy()

print(f"Train={len(raw_train)}  Val={len(raw_val)}  Test={len(raw_test)}")

# ======================================================
# 6. Feature engineering — fit on train only
# ======================================================
def add_cyclic_features(df):
    df = df.copy()
    df['hour']      = df.index.hour
    df['minute']    = df.index.minute
    df['dayofweek'] = df.index.dayofweek
    df['month']     = df.index.month
    df['hour_sin']      = np.sin(2 * np.pi * df['hour']      / 24)
    df['hour_cos']      = np.cos(2 * np.pi * df['hour']      / 24)
    df['minute_sin']    = np.sin(2 * np.pi * df['minute']    / 60)
    df['minute_cos']    = np.cos(2 * np.pi * df['minute']    / 60)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    df['month_sin']     = np.sin(2 * np.pi * (df['month'] - 1) / 12)
    df['month_cos']     = np.cos(2 * np.pi * (df['month'] - 1) / 12)
    return df


def decompose_train(train_series, period=144):
    da = seasonal_decompose(train_series, model='additive',       period=period)
    dm = seasonal_decompose(train_series + 1e-3, model='multiplicative', period=period)
    result = pd.DataFrame({
        'trend_add':    da.trend,
        'seasonal_add': da.seasonal,
        'resid_add':    da.resid,
        'trend_mul':    dm.trend,
        'seasonal_mul': dm.seasonal,
        'resid_mul':    dm.resid,
    }, index=train_series.index)
    return result.bfill().ffill()


def apply_decomp_to_split(df_split, train_decomp):
    period = 144
    result = pd.DataFrame(index=df_split.index)
    n = len(df_split)
    result['trend_add'] = float(train_decomp['trend_add'].iloc[-1])
    result['trend_mul'] = float(train_decomp['trend_mul'].iloc[-1])
    sa = train_decomp['seasonal_add'].values[-period:]
    sm = train_decomp['seasonal_mul'].values[-period:]
    result['seasonal_add'] = np.tile(sa, int(np.ceil(n / period)))[:n]
    result['seasonal_mul'] = np.tile(sm, int(np.ceil(n / period)))[:n]
    result['resid_add'] = (df_split['Power_MW'].values
                           - result['trend_add'].values
                           - result['seasonal_add'].values)
    result['resid_mul'] = (df_split['Power_MW'].values
                           / (result['trend_mul'].values
                              * result['seasonal_mul'].values + 1e-8))
    return result


train_cyc  = add_cyclic_features(raw_train)
val_cyc    = add_cyclic_features(raw_val)
test_cyc   = add_cyclic_features(raw_test)

train_decomp = decompose_train(raw_train['Power_MW'])
val_decomp   = apply_decomp_to_split(raw_val,  train_decomp)
test_decomp  = apply_decomp_to_split(raw_test, train_decomp)

CYCLIC_COLS = ['hour_sin', 'hour_cos', 'minute_sin', 'minute_cos',
               'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos']
DECOMP_COLS = ['trend_add', 'seasonal_add', 'resid_add',
               'trend_mul', 'seasonal_mul', 'resid_mul']

def build_feature_matrix(cyc_df, decomp_df):
    return pd.concat([cyc_df[CYCLIC_COLS], decomp_df[DECOMP_COLS]], axis=1)

feat_train = build_feature_matrix(train_cyc, train_decomp)
feat_val   = build_feature_matrix(val_cyc,   val_decomp)
feat_test  = build_feature_matrix(test_cyc,  test_decomp)

feat_scaler = StandardScaler()
X_train = feat_scaler.fit_transform(feat_train.values)
X_val   = feat_scaler.transform(feat_val.values)
X_test  = feat_scaler.transform(feat_test.values)

y_train = raw_train['Power_MW'].values
y_val   = raw_val['Power_MW'].values
y_test  = raw_test['Power_MW'].values

print(f"Features: {X_train.shape[1]}  |  "
      f"X_train={X_train.shape}  X_val={X_val.shape}  X_test={X_test.shape}")


# ======================================================
# 7. PROPER BCO Core Mathematics
#    Equations numbered as in the paper.
# ======================================================

class BCO_Mussel:
    """
    Generates the hypothetical mussel position Bn.
    Paper Equations 1–9.
    """

    def generate_Bn(self, Xbest: np.ndarray, Up: np.ndarray,
                    Low: np.ndarray, It: int, MaxIt: int, D: int) -> np.ndarray:
        """
        Eq. 1:  Bn = Xbest + H · c · Tempg
        Eq. 2:  H  = 2·((MaxIt - It + 1)/MaxIt)^5 · sin(2π·rand)
        Eq. 5:  g  = randperm(D, N_dims)
        Eq. 6:  N  = ceil(rand·D)  if It < 0.25·MaxIt  else  1
        Eq. 7:  Tempg = (1 - temp)·|Up - Xbest| + temp·|Low - Xbest|
        Eq. 9:  temp_k = round(0.5·(0.5 + rand))  → 0 or 1
        """
        # Eq. 2 — contraction factor H (random, boundary shrinks with iteration)
        H = 2.0 * ((MaxIt - It + 1) / MaxIt) ** 5 * np.sin(2.0 * np.pi * np.random.rand())

        # Eq. 6 — number of dimensions to perturb
        if It < 0.25 * MaxIt:
            N_dims = int(np.ceil(np.random.rand() * D))
        else:
            N_dims = 1

        # Eq. 5 — randomly select N_dims dimensions
        g = np.random.choice(D, size=N_dims, replace=False)

        # Eq. 3–4 — binary selector vector c (1 at selected dims, 0 elsewhere)
        c = np.zeros(D)
        c[g] = 1.0

        # Eq. 8–9 — binary random vector temp (each element 0 or 1 equally)
        temp = np.array([round(0.5 * (0.5 + np.random.rand())) for _ in range(D)])

        # Eq. 7 — Tempg determines search direction from Xbest toward Up or Low
        Tempg = (1.0 - temp) * np.abs(Up - Xbest) + temp * np.abs(Low - Xbest)

        # Eq. 1 — mussel position
        Bn = Xbest + H * c * Tempg

        # Boundary control
        return np.clip(Bn, Low, Up)


class BCO_DPCM:
    """
    Dimensional-Parting-for-Counting Method (DPCM).
    Tracks explored regions per dimension to guide evicted males
    toward the least-explored areas.
    Paper Equations 14–17 and Algorithm 2.
    """

    def __init__(self, D: int, Net: int, Low: np.ndarray, Up: np.ndarray):
        self.D    = D
        self.Net  = Net
        self.Low  = Low
        self.Up   = Up
        # Eq. 15: size of each dimension split
        self.Lnet = (Up - Low) / Net
        # Counting matrix: D rows × Net columns (Algorithm 2, line 1)
        self.crowd = np.zeros((D, Net), dtype=np.int32)

    def update(self, population: np.ndarray):
        """
        Algorithm 2: update counting matrix from all bitterling positions.
        For each bitterling j, for each dimension i:
            find which interval the i-th coordinate falls in, add 1.
        """
        self.crowd = np.zeros((self.D, self.Net), dtype=np.int32)
        N = len(population)
        for j in range(N):
            for i in range(self.D):
                # which Net-interval does position fall in?
                idx = int((population[j, i] - self.Low[i]) / self.Lnet[i])
                idx = np.clip(idx, 0, self.Net - 1)
                self.crowd[i, idx] += 1

    def eviction_position(self) -> np.ndarray:
        """
        Eq. 14:  X_m^i(It+1) = Low + (Indexmin · rand) · Lnet
        Eq. 16:  Indexmin_j  = FMI(crowd_j)  → index of minimum count per dim
        Eq. 17:  rand is a D-dim vector of U(0,1)

        The evicted male is sent to the least-explored region.
        """
        new_pos = np.zeros(self.D)
        rand    = np.random.rand(self.D)
        for i in range(self.D):
            # FMI: index of the interval with fewest visits in dimension i
            Indexmin_i = float(np.argmin(self.crowd[i]))
            new_pos[i] = self.Low[i] + (Indexmin_i * rand[i]) * self.Lnet[i]
        return np.clip(new_pos, self.Low, self.Up)


class BCO_Behaviors:
    """
    The three bitterling search behaviours.
    Paper Equations 10–13.
    """

    def calculate_L(self, It: int, MaxIt: int) -> float:
        """
        Eq. 13:  L = 2·((MaxIt - It + 1)/MaxIt)^0.5 · sin(2π·rand)
        Contraction factor — large early, small later.
        """
        return 2.0 * ((MaxIt - It + 1) / MaxIt) ** 0.5 * np.sin(2.0 * np.pi * np.random.rand())

    def reproductive_behavior_male(self, Bn: np.ndarray, Xm: np.ndarray,
                                   L: float,
                                   Low: np.ndarray, Up: np.ndarray) -> np.ndarray:
        """
        Eq. 10 (Search Style I — male found mussel):
            X_m^i(It+1) = Bn + L·(X_m^i(It) - Bn)
        Male moves toward mussel Bn.
        """
        new_pos = Bn + L * (Xm - Bn)
        return np.clip(new_pos, Low, Up)

    def travel_behavior_male(self, Xm: np.ndarray, L: float,
                             Low: np.ndarray, Up: np.ndarray) -> np.ndarray:
        """
        Eq. 11 (Search Style II — male did NOT find mussel):
            X_m^i(It+1) = X_m^i(It) + L·X_m^i(It)
        Male travels locally (scales its own position).
        """
        new_pos = Xm + L * Xm
        return np.clip(new_pos, Low, Up)

    def female_follow(self, Xm_new: np.ndarray, Xf: np.ndarray,
                      L: float,
                      Low: np.ndarray, Up: np.ndarray) -> np.ndarray:
        """
        Eq. 12 (Female always follows her paired male — all search styles):
            X_f^i(It+1) = X_m^i(It+1) + L·(X_f^i(It) - X_m^i(It+1))
        """
        new_pos = Xm_new + L * (Xf - Xm_new)
        return np.clip(new_pos, Low, Up)


# ======================================================
# 8. Early stopping helper
# ======================================================
def should_stop(history, patience=250, rel_tol=1e-4):
    if len(history) < patience + 1:
        return False
    window = history[-(patience + 1):]
    best, worst = min(window), max(window)
    if abs(worst) < 1e-12:
        return True
    return (worst - best) / abs(worst) < rel_tol


# ======================================================
# 9. Crash-safe checkpoint
# ======================================================
_CRASH_POP = None

def _save_pop_on_crash(sig, frame):
    if _CRASH_POP is not None:
        np.save('bco_population_crash.npy', _CRASH_POP)
        print("💾 Population saved to bco_population_crash.npy")

signal.signal(signal.SIGINT,  _save_pop_on_crash)
signal.signal(signal.SIGTERM, _save_pop_on_crash)


# ======================================================
# 10. RF decoder & evaluator  (Table 3)
# ======================================================
RF_LOW = np.array([10,  0, 1,  2,  1, 0.0, 0,  10, 0.0, 0, 0, 0, 0.0, 0.1],
                  dtype=np.float64)
RF_UP  = np.array([1000, 1, 100, 50, 20, 0.5, 1, 1000, 0.1, 1, 1, 1, 0.1, 1.0],
                  dtype=np.float64)
RF_DIM = len(RF_LOW)


def decode_rf(pos):
    bootstrap   = bool(round(np.clip(pos[9],  0, 1)))
    oob_score   = bool(round(np.clip(pos[10], 0, 1))) if bootstrap else False
    depth_val   = int(np.clip(round(pos[2]),  1, 100))
    leaf_val    = int(np.clip(round(pos[7]),  10, 1000))
    feat_idx    = int(np.clip(round(pos[6]),  0, 1))
    max_samples = float(np.clip(pos[13], 0.1, 1.0)) if bootstrap else None
    if max_samples is not None and max_samples >= 1.0:
        max_samples = None
    return dict(
        n_estimators             = int(np.clip(round(pos[0]),  10, 1000)),
        criterion                = ['squared_error', 'absolute_error'][
                                       int(np.clip(round(pos[1]), 0, 1))],
        max_depth                = None if depth_val >= 100 else depth_val,
        min_samples_split        = int(np.clip(round(pos[3]),  2, 50)),
        min_samples_leaf         = int(np.clip(round(pos[4]),  1, 20)),
        min_weight_fraction_leaf = float(np.clip(pos[5], 0.0, 0.5)),
        max_features             = [None, 'sqrt'][feat_idx],
        max_leaf_nodes           = None if leaf_val >= 1000 else leaf_val,
        min_impurity_decrease    = float(np.clip(pos[8], 0.0, 0.1)),
        bootstrap                = bootstrap,
        oob_score                = oob_score,
        warm_start               = bool(round(np.clip(pos[11], 0, 1))),
        ccp_alpha                = float(np.clip(pos[12], 0.0, 0.1)),
        max_samples              = max_samples,
    )


def evaluate_rf(pos, X_tr, y_tr, X_v, y_v, sub_idx):
    params = decode_rf(pos)
    try:
        rf = RandomForestRegressor(**params, random_state=SEED, n_jobs=-1)
        rf.fit(X_tr[sub_idx], y_tr[sub_idx])
        mae = float(mean_absolute_error(y_v, rf.predict(X_v)))
        return mae, rf
    except Exception:
        print("[RF eval error]"); traceback.print_exc()
        return 1e6, None


# ======================================================
# 11. BCO RF Tuner — PROPER BCO
# ======================================================
class RF_BCO_Tuner:
    """
    Random Forest hyperparameter optimiser using proper BCO.

    Population structure (paper requirement):
        N must be EVEN.
        Males  : pop[0   : N//2]
        Females: pop[N//2: N   ]
        Each male i is paired with female i + N//2.

    Algorithm 3 (paper pseudo-code) implemented faithfully.
    """

    # BCO recommended parameters (paper Sect. 4.1.1)
    PDIS = 0.7   # probability male finds & reaches mussel → RB
    PL   = 0.9   # probability of being evicted           → EB (vs RB)
    NET  = 200   # number of dimension splits for DPCM

    def __init__(self, X_train, y_train, X_val, y_val,
                 pop_size=30, max_iter=700, verbose=True):
        # Ensure even population (BCO requirement)
        if pop_size % 2 != 0:
            pop_size += 1
        self.Xtr, self.ytr = X_train, y_train
        self.Xv,  self.yv  = X_val,   y_val
        self.N        = pop_size
        self.max_iter = max_iter
        self.verbose  = verbose

        self.Low = RF_LOW.copy()
        self.Up  = RF_UP.copy()
        self.D   = RF_DIM

        # BCO components
        self.mussel    = BCO_Mussel()
        self.dpcm      = BCO_DPCM(self.D, self.NET, self.Low, self.Up)
        self.behaviors = BCO_Behaviors()

        self.best_sol   = None
        self.best_fit   = np.inf
        self.best_model = None
        self.history    = []

        rng          = np.random.default_rng(SEED)
        sub_sz       = min(6000, len(X_train))
        self.sub_idx = rng.choice(len(X_train), sub_sz, replace=False)

    def _init_pop(self):
        """Eq. 1 (init): X_i^0 = Low + rand·(Up - Low)"""
        return self.Low + np.random.rand(self.N, self.D) * (self.Up - self.Low)

    def _eval(self, pos):
        return evaluate_rf(pos, self.Xtr, self.ytr,
                           self.Xv, self.yv, self.sub_idx)

    def _checkpoint(self, it, pop):
        state = {'iteration': it, 'best_fit': self.best_fit,
                 'best_sol': self.best_sol.tolist(), 'history': self.history}
        with open('rf_bco_checkpoint.json', 'w') as f:
            json.dump(state, f)
        np.save('rf_bco_pop.npy', pop)

    def optimize(self):
        global _CRASH_POP
        print("\n" + "="*70)
        print("🐟  BCO (Bitterling Colony Optimization) — RF Tuning")
        print(f"    N={self.N}  MaxIt={self.max_iter}  "
              f"Pdis={self.PDIS}  Pl={self.PL}  Net={self.NET}")
        print("="*70)

        # ── Algorithm 3, line 1: Initialise positions randomly ──
        pop = self._init_pop()
        _CRASH_POP = pop

        # ── Algorithm 3, line 2: Initialise counting matrix ──
        self.dpcm.update(pop)

        # ── Algorithm 3, line 3: Calculate fitness & find Xbest ──
        print("  Evaluating initial population...")
        fits, models = [], []
        for ind in pop:
            f, m = self._eval(ind)
            fits.append(f); models.append(m)
        fits = np.array(fits, dtype=np.float64)

        bi = int(np.argmin(fits))
        self.best_sol   = pop[bi].copy()
        self.best_fit   = float(fits[bi])
        self.best_model = models[bi]
        self.history.append(self.best_fit)

        print(f"  Init best MAE={self.best_fit:.4f}")
        print(f"  params={decode_rf(self.best_sol)}")

        # ── Algorithm 3, line 4: Update counting matrix ──
        self.dpcm.update(pop)

        # ── Algorithm 3, line 5: Main loop ──
        for it in range(self.max_iter):

            # Generate mussel Bn (Algorithm 1 / Eqs. 1–9)
            Bn = self.mussel.generate_Bn(
                self.best_sol, self.Up, self.Low, it, self.max_iter, self.D)

            new_pop = pop.copy()

            # ── Algorithm 3, lines 7–17: Male behaviours ──
            for i in range(self.N // 2):
                Xm = pop[i].copy()
                L  = self.behaviors.calculate_L(it, self.max_iter)

                if np.random.rand() < self.PDIS:
                    # Male found mussel
                    if np.random.rand() < self.PL:
                        # Reproductive Behaviour (Eq. 10)
                        new_pos = self.behaviors.reproductive_behavior_male(
                            Bn, Xm, L, self.Low, self.Up)
                    else:
                        # Eviction Behaviour (Eq. 14–17)
                        new_pos = self.dpcm.eviction_position()
                else:
                    # Travel Behaviour (Eq. 11)
                    new_pos = self.behaviors.travel_behavior_male(
                        Xm, L, self.Low, self.Up)

                new_pop[i] = new_pos

            # ── Algorithm 3, lines 18–20: Female behaviours (Eq. 12) ──
            for i in range(self.N // 2, self.N):
                male_idx  = i - self.N // 2   # paired male
                Xm_new    = new_pop[male_idx]
                Xf        = pop[i].copy()
                L         = self.behaviors.calculate_L(it, self.max_iter)
                new_pop[i] = self.behaviors.female_follow(
                    Xm_new, Xf, L, self.Low, self.Up)

            pop = new_pop
            _CRASH_POP = pop

            # ── Algorithm 3, line 21: Boundary check (already done per-move) ──
            pop = np.clip(pop, self.Low, self.Up)

            # ── Algorithm 3, line 22: Update counting matrix ──
            self.dpcm.update(pop)

            # ── Algorithm 3, line 23: Evaluate fitness & update Xbest ──
            fits, models = [], []
            for ind in pop:
                f, m = self._eval(ind)
                fits.append(f); models.append(m)
            fits = np.array(fits, dtype=np.float64)

            bi = int(np.argmin(fits))
            if fits[bi] < self.best_fit:
                self.best_sol   = pop[bi].copy()
                self.best_fit   = float(fits[bi])
                self.best_model = models[bi]
                print(f"  [it={it+1:4d}] 🎉 New best MAE={self.best_fit:.4f}  "
                      f"params={decode_rf(self.best_sol)}")

            self.history.append(self.best_fit)
            self._checkpoint(it, pop)

            if self.verbose:
                print(f"  [it={it+1:4d}] best={self.best_fit:.4f}  "
                      f"Bn_delta={np.linalg.norm(Bn - self.best_sol):.3f}")

            if should_stop(self.history, patience=250, rel_tol=5e-5):
                print(f"  ⏹️  RF BCO early stop @ iter {it+1}")
                break

        p = decode_rf(self.best_sol)
        p['val_mae'] = self.best_fit
        return p


# ======================================================
# 12. Sequence builder
# ======================================================
def make_sequences(data, seq_len):
    Xs, ys = [], []
    for i in range(len(data) - seq_len):
        Xs.append(data[i:i + seq_len])
        ys.append(data[i + seq_len, 0])
    return np.array(Xs), np.array(ys)


# ======================================================
# 13. BiLSTM builder & evaluator  (Table 4)
# ======================================================
BCO_SEQ_LEN   = 36
FINAL_SEQ_LEN = 72
LSTM_SUBSET   = 6000

BILSTM_LOW = np.array([1,  16, 0.1, 0.0, -4.0, 0, 0,  16,  10], dtype=np.float64)
BILSTM_UP  = np.array([4, 256, 0.6, 0.4, -2.0, 1, 1, 256, 150], dtype=np.float64)
BILSTM_DIM = len(BILSTM_LOW)


def decode_bilstm(pos):
    return {
        'n_layers':          int(np.clip(round(pos[0]), 1, 4)),
        'units':             int(np.clip(round(pos[1]), 16, 256)),
        'dropout':           float(np.clip(pos[2], 0.1, 0.6)),
        'recurrent_dropout': float(np.clip(pos[3], 0.0, 0.4)),
        'log_lr':            float(np.clip(pos[4], -4.0, -2.0)),
        'optimizer':         ['adam', 'rmsprop'][int(np.clip(round(pos[5]), 0, 1))],
        'activation':        ['relu', 'tanh'][int(np.clip(round(pos[6]), 0, 1))],
        'batch_size':        int(np.clip(round(pos[7]), 16, 256)),
        'epochs':            int(np.clip(round(pos[8]), 10, 150)),
    }


def _make_bilstm(params, input_shape, use_recurrent_dropout=True, clip_norm=1.0):
    rec_drop = params['recurrent_dropout'] if use_recurrent_dropout else 0.0
    with strategy.scope():
        m = Sequential()
        m.add(tf.keras.layers.Input(shape=input_shape))
        for i in range(params['n_layers']):
            ret = i < params['n_layers'] - 1
            m.add(Bidirectional(
                LSTM(params['units'],
                     return_sequences=ret,
                     activation=params['activation'],
                     recurrent_dropout=rec_drop)
            ))
            if params['dropout'] > 0:
                m.add(Dropout(params['dropout']))
        m.add(Dense(1, activation='linear'))
        lr      = float(10 ** params['log_lr'])
        opt_cls = {'adam': Adam, 'rmsprop': RMSprop}[params['optimizer']]
        opt     = opt_cls(learning_rate=lr, clipnorm=clip_norm)
        m.compile(optimizer=opt, loss='mse', metrics=['mae'])
    return m


def evaluate_bilstm(pos, X_tr, y_tr, X_v, y_v, sub_idx):
    tf.random.set_seed(SEED); np.random.seed(SEED)
    params = decode_bilstm(pos)
    try:
        Xtr_s = X_tr[sub_idx]; ytr_s = y_tr[sub_idx]
        model = _make_bilstm(params,
                             (Xtr_s.shape[1], Xtr_s.shape[2]),
                             use_recurrent_dropout=False)
        es = EarlyStopping(monitor='val_loss', patience=15,
                           restore_best_weights=True, verbose=0)
        model.fit(Xtr_s, ytr_s,
                  batch_size=params['batch_size'],
                  epochs=params['epochs'],
                  validation_data=(X_v, y_v),
                  callbacks=[es], verbose=0)
        mse = float(model.evaluate(X_v, y_v, verbose=0)[0])
        del model; K.clear_session(); gc.collect()
        return mse
    except Exception:
        print("[BiLSTM eval error]"); traceback.print_exc()
        K.clear_session(); gc.collect()
        return 1e6


# ======================================================
# 14. BCO BiLSTM Tuner — PROPER BCO
# ======================================================
class BiLSTM_BCO_Tuner:
    """
    BiLSTM hyperparameter optimiser using proper BCO.
    Identical BCO structure to RF_BCO_Tuner; bounds differ.
    """

    PDIS = 0.7
    PL   = 0.9
    NET  = 200

    def __init__(self, X_train, y_train, X_val, y_val,
                 pop_size=10, max_iter=200, verbose=True):
        if pop_size % 2 != 0:
            pop_size += 1
        self.Xtr, self.ytr = X_train, y_train
        self.Xv,  self.yv  = X_val,   y_val
        self.N        = pop_size
        self.max_iter = max_iter
        self.verbose  = verbose

        self.Low = BILSTM_LOW.copy()
        self.Up  = BILSTM_UP.copy()
        self.D   = BILSTM_DIM

        self.mussel    = BCO_Mussel()
        self.dpcm      = BCO_DPCM(self.D, self.NET, self.Low, self.Up)
        self.behaviors = BCO_Behaviors()

        self.best_sol = None
        self.best_fit = np.inf
        self.history  = []

        rng          = np.random.default_rng(SEED + 7)
        sub_sz       = min(LSTM_SUBSET, len(X_train))
        self.sub_idx = rng.choice(len(X_train), sub_sz, replace=False)

    def _init_pop(self):
        return self.Low + np.random.rand(self.N, self.D) * (self.Up - self.Low)

    def _eval(self, pos):
        return evaluate_bilstm(pos, self.Xtr, self.ytr,
                               self.Xv, self.yv, self.sub_idx)

    def _checkpoint(self, it, pop):
        state = {'iteration': it, 'best_fit': self.best_fit,
                 'best_sol': self.best_sol.tolist(), 'history': self.history}
        with open('bilstm_bco_checkpoint.json', 'w') as f:
            json.dump(state, f)
        np.save('bilstm_bco_pop.npy', pop)

    def optimize(self):
        global _CRASH_POP
        print("\n" + "="*70)
        print("🐟  BCO (Bitterling Colony Optimization) — BiLSTM Tuning")
        print(f"    N={self.N}  MaxIt={self.max_iter}  "
              f"Pdis={self.PDIS}  Pl={self.PL}  Net={self.NET}")
        print("="*70)

        pop = self._init_pop()
        _CRASH_POP = pop
        self.dpcm.update(pop)

        print("  Evaluating initial population...")
        fits = np.array([self._eval(ind) for ind in pop], dtype=np.float64)

        bi = int(np.argmin(fits))
        self.best_sol = pop[bi].copy()
        self.best_fit = float(fits[bi])
        self.history.append(self.best_fit)

        print(f"  Init best MSE={self.best_fit:.6f}")
        print(f"  params={decode_bilstm(self.best_sol)}")

        self.dpcm.update(pop)

        for it in range(self.max_iter):
            Bn = self.mussel.generate_Bn(
                self.best_sol, self.Up, self.Low, it, self.max_iter, self.D)

            new_pop = pop.copy()

            # Male behaviours
            for i in range(self.N // 2):
                Xm = pop[i].copy()
                L  = self.behaviors.calculate_L(it, self.max_iter)

                if np.random.rand() < self.PDIS:
                    if np.random.rand() < self.PL:
                        new_pos = self.behaviors.reproductive_behavior_male(
                            Bn, Xm, L, self.Low, self.Up)
                    else:
                        new_pos = self.dpcm.eviction_position()
                else:
                    new_pos = self.behaviors.travel_behavior_male(
                        Xm, L, self.Low, self.Up)

                new_pop[i] = new_pos

            # Female behaviours
            for i in range(self.N // 2, self.N):
                male_idx   = i - self.N // 2
                Xm_new     = new_pop[male_idx]
                Xf         = pop[i].copy()
                L          = self.behaviors.calculate_L(it, self.max_iter)
                new_pop[i] = self.behaviors.female_follow(
                    Xm_new, Xf, L, self.Low, self.Up)

            pop = np.clip(new_pop, self.Low, self.Up)
            _CRASH_POP = pop

            self.dpcm.update(pop)

            fits = np.array([self._eval(ind) for ind in pop], dtype=np.float64)
            bi   = int(np.argmin(fits))

            if fits[bi] < self.best_fit:
                self.best_sol = pop[bi].copy()
                self.best_fit = float(fits[bi])
                print(f"  [it={it+1:3d}] 🎉 New best MSE={self.best_fit:.6f}  "
                      f"params={decode_bilstm(self.best_sol)}")

            self.history.append(self.best_fit)
            self._checkpoint(it, pop)

            if self.verbose:
                print(f"  [it={it+1:3d}/{self.max_iter}] "
                      f"best={self.best_fit:.6f}")

            if should_stop(self.history, patience=50, rel_tol=1e-4):
                print(f"  ⏹️  BiLSTM BCO early stop @ iter {it+1}")
                break

        p = decode_bilstm(self.best_sol)
        p['val_mse'] = self.best_fit
        return p


# ======================================================
# 15. RF prediction uncertainty (std across trees)
# ======================================================
def rf_prediction_std(rf_model, X):
    tree_preds = np.stack([t.predict(X) for t in rf_model.estimators_], axis=1)
    return tree_preds.mean(axis=1), tree_preds.std(axis=1)


# ======================================================
# 16. Main
# ======================================================
if __name__ == "__main__":

    # ── STAGE 1: BCO → RF ──────────────────────────────
    rf_tuner  = RF_BCO_Tuner(X_train, y_train, X_val, y_val,
                              pop_size=30, max_iter=700)
    best_rf_p = rf_tuner.optimize()

    print("\n" + "="*70)
    print("🌟  BEST RF HYPERPARAMETERS  (BCO)")
    print("="*70)
    for k, v in best_rf_p.items():
        print(f"  {k:>28}: {v}")

    # Re-train RF on full train+val
    rf_params = {k: v for k, v in best_rf_p.items() if k != 'val_mae'}
    X_tr_val  = np.vstack([X_train, X_val])
    y_tr_val  = np.concatenate([y_train, y_val])
    rf_final  = RandomForestRegressor(**rf_params, random_state=SEED, n_jobs=-1)
    rf_final.fit(X_tr_val, y_tr_val)

    y_pred_rf_train          = rf_final.predict(X_train)
    y_pred_rf_val            = rf_final.predict(X_val)
    y_pred_rf_test, y_std_rf = rf_prediction_std(rf_final, X_test)

    print(f"\n  RF test MAE : {mean_absolute_error(y_test, y_pred_rf_test):.4f}")
    print(f"  RF test R²  : {r2_score(y_test, y_pred_rf_test):.4f}")

    # ── STAGE 2: Residuals (no leakage) ────────────────
    residuals_train = y_train - y_pred_rf_train
    resid_deriv_tr  = np.diff(residuals_train, prepend=residuals_train[0])
    residuals_val   = y_val - y_pred_rf_val
    resid_deriv_val = np.diff(residuals_val, prepend=residuals_val[0])

    resid_scaler = MinMaxScaler((-1, 1))
    deriv_scaler = MinMaxScaler((-1, 1))
    res_tr_sc  = resid_scaler.fit_transform(residuals_train.reshape(-1, 1))
    der_tr_sc  = deriv_scaler.fit_transform(resid_deriv_tr.reshape(-1, 1))
    lstm_in_tr = np.hstack([res_tr_sc, der_tr_sc])

    res_val_sc  = resid_scaler.transform(residuals_val.reshape(-1, 1))
    der_val_sc  = deriv_scaler.transform(resid_deriv_val.reshape(-1, 1))
    lstm_in_val = np.hstack([res_val_sc, der_val_sc])

    # ── STAGE 3: Sequences ─────────────────────────────
    X_seq_bco_tr,  y_seq_bco_tr  = make_sequences(lstm_in_tr,  BCO_SEQ_LEN)
    X_seq_bco_val, y_seq_bco_val = make_sequences(lstm_in_val, BCO_SEQ_LEN)
    X_seq_fin_tr,  y_seq_fin_tr  = make_sequences(lstm_in_tr,  FINAL_SEQ_LEN)
    X_seq_fin_val, y_seq_fin_val = make_sequences(lstm_in_val, FINAL_SEQ_LEN)

    print(f"\n  BCO LSTM   train={len(X_seq_bco_tr):5d}  val={len(X_seq_bco_val):5d}")
    print(f"  FINAL LSTM train={len(X_seq_fin_tr):5d}  val={len(X_seq_fin_val):5d}")

    # ── STAGE 4: BCO → BiLSTM ──────────────────────────
    bilstm_tuner = BiLSTM_BCO_Tuner(
        X_seq_bco_tr, y_seq_bco_tr,
        X_seq_bco_val, y_seq_bco_val,
        pop_size=10, max_iter=200)
    best_lstm_p = bilstm_tuner.optimize()

    print("\n" + "="*70)
    print("🌟  BEST BiLSTM HYPERPARAMETERS  (BCO)")
    print("="*70)
    for k, v in best_lstm_p.items():
        print(f"  {k:>22}: {v}")

    # ── STAGE 5: Final BiLSTM training ─────────────────
    print("\n  Training final BiLSTM...")
    final_model = _make_bilstm(
        best_lstm_p,
        input_shape=(X_seq_fin_tr.shape[1], X_seq_fin_tr.shape[2]),
        use_recurrent_dropout=True,
        clip_norm=1.0
    )
    es_fin  = EarlyStopping(monitor='val_loss', patience=25,
                            restore_best_weights=True, verbose=1)
    rlr_fin = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=10, min_lr=1e-6, verbose=1)
    ckpt    = ModelCheckpoint('best_bilstm.keras',
                              monitor='val_loss', save_best_only=True, verbose=0)
    final_epochs = min(best_lstm_p['epochs'] * 3, 500)
    final_model.fit(
        X_seq_fin_tr, y_seq_fin_tr,
        batch_size=best_lstm_p['batch_size'],
        epochs=final_epochs,
        validation_data=(X_seq_fin_val, y_seq_fin_val),
        callbacks=[es_fin, rlr_fin, ckpt],
        verbose=1,
    )
    final_model.save('final_best_model.keras')
    print("  ✅ Saved final_best_model.keras")

    # ── STAGE 6: TEST evaluation ───────────────────────
    residuals_test = y_test - y_pred_rf_test
    resid_deriv_te = np.diff(residuals_test, prepend=residuals_test[0])
    res_te_sc      = resid_scaler.transform(residuals_test.reshape(-1, 1))
    der_te_sc      = deriv_scaler.transform(resid_deriv_te.reshape(-1, 1))
    lstm_in_te     = np.hstack([res_te_sc, der_te_sc])
    X_seq_te, _    = make_sequences(lstm_in_te, FINAL_SEQ_LEN)

    pred_scaled = final_model.predict(X_seq_te, verbose=0).flatten()
    pred_resid  = resid_scaler.inverse_transform(
                      pred_scaled.reshape(-1, 1)).flatten()

    n_out  = len(pred_resid)
    yt     = y_test[-n_out:]
    yrf    = y_pred_rf_test[-n_out:]
    ycomb  = yrf + pred_resid

    def m3(yt, yp):
        return (mean_absolute_error(yt, yp),
                np.sqrt(mean_squared_error(yt, yp)),
                r2_score(yt, yp))

    mae_rf,  rmse_rf,  r2_rf  = m3(yt, yrf)
    mae_cb,  rmse_cb,  r2_cb  = m3(yt, ycomb)

    err_rf   = np.abs(yt - yrf)
    err_comb = np.abs(yt - ycomb)
    try:
        stat, pval = wilcoxon(err_rf, err_comb)
        sig_str    = f"Wilcoxon stat={stat:.1f}  p={pval:.4f}"
    except Exception:
        sig_str = "Wilcoxon test failed (identical arrays?)"

    print("\n" + "="*70)
    print("📊  FINAL TEST RESULTS  (BCO)")
    print("="*70)
    print(f"  {'Metric':<10} {'RF Only':>12} {'RF+BiLSTM':>12}")
    print("  " + "-"*36)
    print(f"  {'MAE':<10} {mae_rf:>12.4f} {mae_cb:>12.4f}")
    print(f"  {'RMSE':<10} {rmse_rf:>12.4f} {rmse_cb:>12.4f}")
    print(f"  {'R²':<10} {r2_rf:>12.4f} {r2_cb:>12.4f}")
    print(f"\n  {sig_str}")
    print("="*70)

    # ── STAGE 7: Convergence diagnostics plot ──────────
    n_plot = min(500, n_out)
    fig, axes = plt.subplots(4, 1, figsize=(14, 16))

    axes[0].plot(yt[:n_plot],    label='Actual',       lw=1)
    axes[0].plot(yrf[:n_plot],   label='RF (BCO)',      lw=1, alpha=0.8)
    axes[0].plot(ycomb[:n_plot], label='RF+BiLSTM (BCO)', lw=1, alpha=0.8)
    axes[0].fill_between(range(n_plot),
                         yrf[:n_plot] - y_std_rf[-n_out:][:n_plot],
                         yrf[:n_plot] + y_std_rf[-n_out:][:n_plot],
                         alpha=0.15, label='RF ±1σ')
    axes[0].set_title('Predictions vs Actual — first 500 test samples')
    axes[0].set_ylabel('Power (MW)'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(rf_tuner.history, 'o-', ms=3, color='steelblue')
    axes[1].set_title('RF BCO Convergence  '
                      r'($H=2\cdot((MaxIt-It+1)/MaxIt)^5\cdot\sin(2\pi r)$)')
    axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Best val MAE')
    axes[1].grid(alpha=0.3)

    axes[2].plot(bilstm_tuner.history, 'o-', ms=3, color='darkorange')
    axes[2].set_title('BiLSTM BCO Convergence')
    axes[2].set_xlabel('Iteration'); axes[2].set_ylabel('Best val MSE')
    axes[2].grid(alpha=0.3)

    axes[3].plot(err_rf[:n_plot],   label='|RF error|',        lw=1, alpha=0.8)
    axes[3].plot(err_comb[:n_plot], label='|RF+BiLSTM error|', lw=1, alpha=0.8)
    axes[3].set_title('Absolute Error Comparison')
    axes[3].set_xlabel('Sample'); axes[3].set_ylabel('|Error| (MW)')
    axes[3].legend(); axes[3].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('results_BCO_proper.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("  ✅ results_BCO_proper.png saved")
    print("\n  ✅ BCO (Bitterling Colony Optimization) completed successfully.")